Actividad práctica 10: “EmotionLearn” App para analizar los sentimientos en opiniones sobre clases Objetivo: Diseñar una app educativa que detecte si el sentimiento de una opinión escrita por un estudiante es positiva, negativa o neutral, usando NLP y modelos de clasificación. Alcance: Permite a los docentes analizar el tono emcional de los comentarios de sus estudiantes, con el fin de ajustar sus estrategias didácticas. Producto final: App interactiva en Google Colab donde el usuario ingresa una opinión y obtiene el análisis de sentimiento.

In [ ]:
# Paso 1: Importar librerías necesarias
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from ipywidgets import Text, Output, VBox, Button
from IPython.display import display

# Paso 2: Dataset simulado
opiniones = [
    "Me encantó la clase, todo fue claro y bien explicado",
    "No entendí nada, fue una pérdida de tiempo",
    "Estuvo regular, algunas partes fueron interesantes",
    "Excelente docente, aprendí mucho",
    "Muy aburrido, no volvería a tomar esta materia",
    "No estuvo tan mal, aunque podría mejorar"
]
etiquetas = ['positivo', 'negativo', 'neutral', 'positivo', 'negativo', 'neutral']

# Paso 3: Preparar los datos
tokenizer = Tokenizer(num_words=1000, oov_token='<OOV>')
tokenizer.fit_on_texts(opiniones)
seqs = tokenizer.texts_to_sequences(opiniones)
padded = pad_sequences(seqs, padding='post')

label_map = {'positivo': 0, 'negativo': 1, 'neutral': 2}
y = tf.keras.utils.to_categorical([label_map[e] for e in etiquetas], num_classes=3)

# Paso 4: Definir y entrenar el modelo
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(1000, 16, input_length=padded.shape[1]),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(padded, y, epochs=100, verbose=0)

# Paso 5: Interfaz interactiva
entrada = Text(description='Opinión:')
out = Output()
boton = Button(description='Analizar sentimiento')

rev_map = {v: k for k, v in label_map.items()}

def analizar_sentimiento(b):
    texto = entrada.value
    seq = tokenizer.texts_to_sequences([texto])
    pad = pad_sequences(seq, maxlen=padded.shape[1], padding='post')
    pred = model.predict(pad)[0]
    resultado = rev_map[np.argmax(pred)]
    with out:
        out.clear_output()
        print(f"Sentimiento detectado: {resultado}")

boton.on_click(analizar_sentimiento)
display(VBox([entrada, boton, out]))

Actividad práctica 11: “OrientaTec” App para recomendar campos de estudio basado en intereses.

Objetivo: Diseñar una app que permita recomendar áreas académicas (ciencias, ingeniería, negocios, salud, diseño, etc) en función de los intereses personales del estudiante.

Alcance: Puede aplicarse en talleres de orientación vocacional en preparatorias o en programas de ingreso universitario o como parte de un módulo de orientación educativa.

In [ ]:
# Paso 1: Importar librerías
import tensorflow as tf
import numpy as np
import pandas as pd
from ipywidgets import Checkbox, Button, Output, VBox
from IPython.display import display

# Paso 2: Crear dataset simulado
intereses = ['resolver problemas', 'leer y escribir', 'ayudar a otros', 'diseñar cosas', 'liderar proyectos', 'hacer experimentos']
datos = []
etiquetas = []

for _ in range(300):
    perfil = np.random.choice([0, 1], size=6)
    datos.append(perfil)
    if perfil[0] == 1 and perfil[5] == 1:
        etiquetas.append('Ciencias')
    elif perfil[1] == 1 and perfil[3] == 1:
        etiquetas.append('Diseño')
    elif perfil[2] == 1:
        etiquetas.append('Salud')
    elif perfil[4] == 1:
        etiquetas.append('Negocios')
    else:
        etiquetas.append('General')

data = pd.DataFrame(datos, columns=intereses)
y_map = {'Ciencias': 0, 'Diseño': 1, 'Salud': 2, 'Negocios': 3, 'General': 4}
y = tf.keras.utils.to_categorical([y_map[e] for e in etiquetas], num_classes=5)

# Paso 3: Entrenamiento del modelo
X = data.values
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(6,)),
    tf.keras.layers.Dense(5, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=0)

# Paso 4: Interfaz interactiva
out = Output()
checkboxes = [Checkbox(description=i) for i in intereses]
boton = Button(description='Recomendar campo de estudio')

rev_map = {v: k for k, v in y_map.items()}

def recomendar(b):
    entrada = np.array([[int(cb.value) for cb in checkboxes]])
    pred = model.predict(entrada)[0]
    campo = rev_map[np.argmax(pred)]
    with out:
        out.clear_output()
        print(f"Campo de estudio recomendado: {campo}")

boton.on_click(recomendar)
display(VBox(checkboxes + [boton, out]))

Actividad práctica 13: “LearnStyle” Detección de estilo de aprendizaje basado en respuestas a cuestionarios

Objetivo: Desarrollar una app que prediga el estilo de aprendizaje domiinante (visual, auditivo, kinestésico o mixto) a partir de las respuestas de un estudiante a un breve cuestionario.

Alcance: Es útil para orientar las estrategias de enseñanza del docente al inicio de un curso o como parte de un programa de tutoría académica.

Producto final: Interfaz interactiva de selección de opciones tipo checkbox que clasifica automáticamente el estilo de aprendizaje predominante.

In [ ]:
# Paso 1: Importar bibliotecas
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.utils import to_categorical
from ipywidgets import Checkbox, Button, VBox, Output
from IPython.display import display

# Paso 2: Crear dataset simulado
preguntas = ['Prefiero diagramas', 'Escucho podcasts', 'Aprendo moviéndome',
             'Hago mapas mentales', 'Recuerdo lo que escucho', 'Necesito tocar']
datos = []
etiquetas = []

for _ in range(300):
    perfil = np.random.choice([0, 1], size=6)
    datos.append(perfil)
    if perfil[0] + perfil[3] > 1:
        etiquetas.append('Visual')
    elif perfil[1] + perfil[4] > 1:
        etiquetas.append('Auditivo')
    elif perfil[2] + perfil[5] > 1:
        etiquetas.append('Kinestésico')
    else:
        etiquetas.append('Mixto')

data = pd.DataFrame(datos, columns=preguntas)
y_map = {'Visual': 0, 'Auditivo': 1, 'Kinestésico': 2, 'Mixto': 3}
y = to_categorical([y_map[e] for e in etiquetas], num_classes=4)
X = data.values

# Paso 3: Modelo TensorFlow
model = tf.keras.Sequential([
    tf.keras.layers.Dense(12, activation='relu', input_shape=(6,)),
    tf.keras.layers.Dense(4, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=150, verbose=0)

# Paso 4: Interfaz interactiva
out = Output()
checkboxes = [Checkbox(description=p) for p in preguntas]
boton = Button(description='Detectar estilo')

rev_map = {v: k for k, v in y_map.items()}

def detectar(b):
    entrada = np.array([[int(cb.value) for cb in checkboxes]])
    pred = model.predict(entrada)[0]
    estilo = rev_map[np.argmax(pred)]
    with out:
        out.clear_output()
        print(f"Estilo de aprendizaje dominante: {estilo}")

boton.on_click(detectar)
display(VBox(checkboxes + [boton, out]))

Actividad práctica 27: Chatbot sobre estado emocional y psicológico

Objetivo: Diseñar un chatbot interactivo uqe permita a los estudiantes reflexionar sobre su estado emocional y psicológico en el contexto académico, brindando retroalimentación automática y servicios de apoyo.

Alcance: cuestionario breve con preguntas tipo test, evaluación de posibles señales de estrés, ansiedad, desmotivación o fatiga académica. Recomendaciones personalizadas de acuerdo a los resultados.

Producto final: un chatbot con interfaz gráfica en Google Colab que simula una entrevista de bienestar psicológico básico, útil para evaluación temprana del estado emocional de un estudiante.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import datetime

# --- Preguntas específicas para estudiantes de ingeniería ---
preguntas = [
    {
        "pregunta": "¿Sientes que los proyectos o tareas de ingeniería te sobrepasan?",
        "opciones": {
            "No, los gestiono bien": 0,
            "A veces me estresan": 1,
            "Frecuentemente me agotan": 2,
            "Casi siempre colapsan mis tiempos": 3
        }
    },
    {
        "pregunta": "¿Te cuesta mantener la concentración al programar o resolver problemas?",
        "opciones": {
            "Para nada": 0,
            "Solo cuando estoy cansado": 1,
            "Sí, muy seguido": 2,
            "Siempre me distraigo": 3
        }
    },
    {
        "pregunta": "¿Has sentido frustración al no entender temas de matemáticas o física?",
        "opciones": {
            "No, los entiendo bien": 0,
            "A veces me frustro": 1,
            "Frecuentemente me frustro": 2,
            "Estoy frustrado constantemente": 3
        }
    },
    {
        "pregunta": "¿Duermes menos de 6 horas por preparar tareas o estudiar?",
        "opciones": {
            "No, duermo bien": 0,
            "Ocasionalmente": 1,
            "Casi todos los días": 2,
            "Siempre sacrifico el sueño": 3
        }
    },
    {
        "pregunta": "¿Sientes que estudiar ingeniería te está agotando emocionalmente?",
        "opciones": {
            "No, me apasiona": 0,
            "Un poco": 1,
            "Sí, bastante": 2,
            "Estoy al límite": 3
        }
    }
]

# Variables
puntuacion_total = 0
puntuaciones_pregunta = []
indice_pregunta = 0
salida = widgets.Output()

# Mostrar pregunta actual
def mostrar_pregunta(indice):
    salida.clear_output()
    with salida:
        print(f"Pregunta {indice + 1} de {len(preguntas)}:\n{preguntas[indice]['pregunta']}")
        botones = []
        for texto, valor in preguntas[indice]["opciones"].items():
            btn = widgets.Button(description=texto, layout=widgets.Layout(width='auto'))
            def on_click(b, val=valor):
                procesar_respuesta(val)
            btn.on_click(on_click)
            botones.append(btn)
        display(widgets.VBox(botones))

# Procesar respuesta y avanzar
def procesar_respuesta(valor):
    global puntuacion_total, indice_pregunta
    puntuacion_total += valor
    puntuaciones_pregunta.append(valor)
    indice_pregunta += 1
    if indice_pregunta < len(preguntas):
        mostrar_pregunta(indice_pregunta)
    else:
        mostrar_resultado()

# Mostrar gráfico
def mostrar_grafico():
    etiquetas = [f"P{i+1}" for i in range(len(preguntas))]
    plt.figure(figsize=(8,4))
    plt.bar(etiquetas, puntuaciones_pregunta, color='steelblue')
    plt.title("Evaluación emocional por pregunta")
    plt.xlabel("Preguntas")
    plt.ylabel("Nivel de afectación")
    plt.ylim(0, 3.5)
    plt.grid(axis='y')
    plt.show()

# Mostrar resultado final
def mostrar_resultado():
    salida.clear_output()
    with salida:
        print("🧠 Evaluación emocional completada.\n")
        print(f"🧮 Puntaje total: {puntuacion_total} / {len(preguntas)*3}\n")

        if puntuacion_total <= 4:
            estado = "✅ Estado emocional estable."
            recomendaciones = [
                "Sigue con tu ritmo actual, pero no descuides tu bienestar.",
                "Sigue usando técnicas de organización y estudio eficiente."
            ]
        elif 5 <= puntuacion_total <= 8:
            estado = "⚠️ Signos leves de agotamiento académico."
            recomendaciones = [
                "Intenta dividir tus proyectos en tareas pequeñas.",
                "Haz pausas activas y no olvides descansar."
            ]
        elif 9 <= puntuacion_total <= 12:
            estado = "⚠️ Estado de estrés académico moderado."
            recomendaciones = [
                "Apóyate en tus profesores o tutores para organizar tus materias.",
                "Practica técnicas de respiración y usa planificadores semanales.",
                "Evita estudiar de madrugada."
            ]
        else:
            estado = "🚨 Riesgo alto de agotamiento o ansiedad académica."
            recomendaciones = [
                "Contacta con un orientador o profesional de salud mental.",
                "Considera reducir temporalmente tu carga académica.",
                "Recuerda que pedir ayuda también es parte del proceso de ingeniería."
            ]

        print(estado)
        print("\n📌 Recomendaciones:")
        for rec in recomendaciones:
            print("-", rec)

        # Recursos sugeridos
        print("\n📚 Recursos sugeridos para estudiantes de ingeniería:")
        print("- Canal de YouTube: 'Unicoos' (física/matemáticas)")
        print("- Técnica Pomodoro (25 min estudio + 5 descanso)")
        print("- App gratuita: Focus To-Do, Notion, Forest")
        print("- Consulta con tu tutor académico o centro de bienestar universitario\n")

        mostrar_grafico()

        # Exportar resumen sin error
        fecha = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
        resumen_recomendaciones = "\n- " + "\n- ".join(recomendaciones)
        resumen = (
            f"Evaluación psicológica - Estudiantes de Ingeniería\n"
            f"Fecha: {fecha}\n"
            f"Puntaje total: {puntuacion_total} / {len(preguntas)*3}\n"
            f"Estado: {estado}\n"
            f"Recomendaciones:{resumen_recomendaciones}"
        )

        print("📝 Resumen:")
        print(resumen)

# Iniciar
mostrar_pregunta(indice_pregunta)
display(salida)

Actividad práctica 21: Chatbot para interpretar gráficos estadísticos

Objetivo: Apoyar al estudiante a interpretar gráficos estadísticos básicos de barra o de pastel, identificando tendencias o datos claves.

Alcance: Generación del gráfico con matplotlib, pregunta relacionada con el gráfico y retroalimentación automática sobre la respuesta del usuario.

Producto final: una app interactiva en Google Colab que muestra un gráfico estadístico, pregunta algo sobre él y valida la repsuesta del estudiante.

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Datos
categorias = ['A', 'B', 'C', 'D']
valores = [5, 8, 6, 3]
respuesta_correcta = 'B'

# Gráfico
def graficar():
    plt.figure(figsize=(6,4))
    plt.bar(categorias, valores, color='skyblue')
    plt.title("Estudiantes por grupo")
    plt.xlabel("Grupo")
    plt.ylabel("Cantidad")
    plt.show()

# Widgets
entrada = widgets.Text(placeholder="¿Qué grupo tiene más estudiantes?")
boton = widgets.Button(description='Verificar')
salida = widgets.Output()

def verificar(b):
    r = entrada.value.strip().upper()
    with salida:
        clear_output()
        graficar()
        if r == respuesta_correcta:
            print("✅ ¡Correcto! El grupo B tiene 8 estudiantes.")
        else:
            print(f"❌ Incorrecto. La respuesta correcta es '{respuesta_correcta}'.")

# Mostrar interfaz
with salida:
    graficar()
display(salida, entrada, boton)
boton.on_click(verificar)

Actividad práctica 22: Chatbot sobre cálculo de áreas de figuras geométricas

Objetivo: Practicar el cálculo de áreas de figuras geométricas con retroalimentación automatizada entiempo real, registre el desempeño del estudiante, ajuste la dificultad y guía paso a paso de la solución.

Alcance: solo para figuras geométricas básicas, tales como: triángulo, cuadrado, rectángulo, círculo y trapecio.

Producto final: una app interactiva en Google Colab con una interfaz visual, que verifique los resultados de forma automática, explique paso a paso la solución y genere infinitos ejercicios con diferentes valores.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random, math, time

salida = widgets.Output()

# Figuras por nivel

def generar_triangulo():
    base = random.randint(4, 10)
    altura = random.randint(3, 8)
    area = round((base * altura) / 2, 2)
    pasos = f"Área = (base * altura) / 2 = ({base} * {altura}) / 2 = {area}"
    return "Triángulo", f"Base = {base}, Altura = {altura}", area, pasos

def generar_cuadrado():
    lado = random.randint(3, 10)
    area = round(lado ** 2, 2)
    pasos = f"Área = lado² = {lado}² = {area}"
    return "Cuadrado", f"Lado = {lado}", area, pasos

def generar_rectangulo():
    base = random.randint(4, 10)
    altura = random.randint(3, 8)
    area = round(base * altura, 2)
    pasos = f"Área = base * altura = {base} * {altura} = {area}"
    return "Rectángulo", f"Base = {base}, Altura = {altura}", area, pasos

def generar_circulo():
    radio = random.randint(2, 6)
    area = round(math.pi * radio ** 2, 2)
    pasos = f"Área = π * r² = π * {radio}² = π * {radio**2} ≈ {area}"
    return "Círculo", f"Radio = {radio}", area, pasos

def generar_trapecio():
    base1 = random.randint(3, 7)
    base2 = random.randint(4, 9)
    altura = random.randint(3, 6)
    area = round(((base1 + base2) * altura) / 2, 2)
    pasos = f"Área = ((base1 + base2) * altura) / 2 = (({base1} + {base2}) * {altura}) / 2 = {area}"
    return "Trapecio", f"Base1 = {base1}, Base2 = {base2}, Altura = {altura}", area, pasos

# Niveles
niveles = {
    1: [generar_triangulo, generar_cuadrado],
    2: [generar_rectangulo],
    3: [generar_circulo, generar_trapecio]
}

# Variables globales

nivel = 1
aciertos_consecutivos = 0
puntuacion = 0
tiempos = []
ejercicio_actual = {}
inicio_tiempo = 0

# Widgets

entrada_area = widgets.Text(placeholder="Ingresa el área")
btn_verificar = widgets.Button(description="✅ Verificar")
btn_pasos = widgets.Button(description="📘 Ver pasos")
btn_siguiente = widgets.Button(description="🔄 Siguiente")
lbl_estado = widgets.Label()

# Lógica principal

def nuevo_ejercicio(_=None):
    global ejercicio_actual, inicio_tiempo
    gen = random.choice(niveles[nivel])
    nombre, datos, area, pasos = gen()
    ejercicio_actual = {
        "nombre": nombre,
        "datos": datos,
        "area": area,
        "pasos": pasos
    }
    entrada_area.value = ""
    inicio_tiempo = time.time()
    salida.clear_output()
    with salida:
        print(f"📐 Figura: {nombre}")
        print(f"🔢 Datos: {datos}")

def verificar(_):
    global puntuacion, aciertos_consecutivos, nivel
    try:
        respuesta = float(entrada_area.value)
        tiempo = round(time.time() - inicio_tiempo, 2)
        tiempos.append(tiempo)
        salida.clear_output()
        with salida:
            print(f"📐 Figura: {ejercicio_actual['nombre']}")
            print(f"🔢 Datos: {ejercicio_actual['datos']}")
            print(f"⏱️ Tiempo de respuesta: {tiempo} s")
            if abs(respuesta - ejercicio_actual['area']) < 0.5:
                puntuacion += 10
                aciertos_consecutivos += 1
                print("✅ ¡Correcto!")
                if aciertos_consecutivos >= 3 and nivel < 3:
                    nivel += 1
                    print(f"🚀 Nivel aumentado a {nivel}")
                    aciertos_consecutivos = 0
            else:
                aciertos_consecutivos = 0
                print(f"❌ Incorrecto. El área correcta es ≈ {ejercicio_actual['area']}")
            lbl_estado.value = f"🎯 Puntos: {puntuacion} | Nivel: {nivel} | Tiempo promedio: {round(sum(tiempos)/len(tiempos), 2)} s"
    except:
        with salida:
            print("⚠️ Ingresa un número válido.")

def mostrar_pasos(_):
    with salida:
        print("📘 Pasos para resolver:")
        print(ejercicio_actual['pasos'])

# Conexión de eventos

btn_verificar.on_click(verificar)
btn_pasos.on_click(mostrar_pasos)
btn_siguiente.on_click(nuevo_ejercicio)

# Interfaz

display(salida)
display(entrada_area)
display(widgets.HBox([btn_verificar, btn_pasos, btn_siguiente]))
display(lbl_estado)

nuevo_ejercicio()

Actividad práctica 23: Chatbot de análisis vocacional

Objetivo: Guiar al estudiante en la exploración de su perfil vocacional mediante un test interactivo simple con retroalimentación inmediata.

Alcance: uso de ipywidgets para preguntas dinámicas, registro de respuestas en tiempo real, clasificación automática en 4 perfiles (científico, técnico, artístico, social), retraolimentación detallada y amigable.

Producto final: una app interactiva que presente una pregunta a la vez, al final muestre el perfil vocacional predominante y recomendaciones, opción de volver a empezar.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Preguntas y respuestas
preguntas = [
    ("¿Qué actividad disfrutas más?", ["Leer artículos científicos", "Pintar o dibujar", "Reparar cosas", "Ayudar a personas"]),
    ("¿Qué asignatura prefieres?", ["Matemáticas", "Arte", "Tecnología", "Psicología"]),
    ("¿Cómo te gusta trabajar?", ["Analizando datos", "Creando cosas nuevas", "Con herramientas", "En equipo con personas"]),
    ("¿Qué hobby te interesa más?", ["Resolver acertijos", "Tocar un instrumento", "Armar circuitos", "Voluntariado"]),
    ("¿Qué valoras más en un trabajo?", ["Descubrimiento", "Expresión", "Precisión", "Impacto social"]),
    ("¿Con qué palabra te identificas más?", ["Lógico", "Creativo", "Práctico", "Empático"])
]

# Correspondencia para perfiles
perfil_map = {
    "Leer artículos científicos": "científico",
    "Matemáticas": "científico",
    "Analizando datos": "científico",
    "Resolver acertijos": "científico",
    "Descubrimiento": "científico",
    "Lógico": "científico",

    "Pintar o dibujar": "artístico",
    "Arte": "artístico",
    "Creando cosas nuevas": "artístico",
    "Tocar un instrumento": "artístico",
    "Expresión": "artístico",
    "Creativo": "artístico",

    "Reparar cosas": "técnico",
    "Tecnología": "técnico",
    "Con herramientas": "técnico",
    "Armar circuitos": "técnico",
    "Precisión": "técnico",
    "Práctico": "técnico",

    "Ayudar a personas": "social",
    "Psicología": "social",
    "En equipo con personas": "social",
    "Voluntariado": "social",
    "Impacto social": "social",
    "Empático": "social"
}

# Resultados y recomendaciones
recomendaciones = {
    "científico": "🔬 Perfil Científico: Podrías destacar en áreas como Física, Matemáticas, Biología, Ingeniería o Investigación.",
    "artístico": "🎨 Perfil Artístico: Podrías sobresalir en Diseño, Música, Artes Visuales, Publicidad o Cine.",
    "técnico": "🔧 Perfil Técnico: Carreras como Mecatrónica, Robótica, Sistemas o Mantenimiento son una excelente opción.",
    "social": "👥 Perfil Social: Psicología, Educación, Trabajo Social o Comunicación podrían ser tu vocación."
}

# Control de flujo
indice_pregunta = 0
respuestas = []
conteo = {"científico": 0, "artístico": 0, "técnico": 0, "social": 0}

# Widgets
label_pregunta = widgets.Label()
radio_respuesta = widgets.RadioButtons()
boton_siguiente = widgets.Button(description="Siguiente")
barra = widgets.IntProgress(min=0, max=len(preguntas), value=0)
salida = widgets.Output()

def mostrar_pregunta():
    label_pregunta.value = preguntas[indice_pregunta][0]
    radio_respuesta.options = preguntas[indice_pregunta][1]
    barra.value = indice_pregunta

def siguiente(_):
    global indice_pregunta
    seleccion = radio_respuesta.value
    respuestas.append(seleccion)
    perfil = perfil_map.get(seleccion)
    if perfil:
        conteo[perfil] += 1

    indice_pregunta += 1
    if indice_pregunta < len(preguntas):
        mostrar_pregunta()
    else:
        mostrar_resultado()

def mostrar_resultado():
    salida.clear_output()
    with salida:
        print("✅ Test completado.\n")
        perfil_final = max(conteo, key=conteo.get)
        print(f"🔎 Tu perfil vocacional dominante es: **{perfil_final.upper()}**\n")
        print(recomendaciones[perfil_final])
        print("\nRespuestas seleccionadas:")
        for i, r in enumerate(respuestas):
            print(f"{i+1}. {preguntas[i][0]} → {r}")
    boton_siguiente.disabled = True

# Botón reinicio (opcional)
def reiniciar(_):
    global indice_pregunta, respuestas, conteo
    indice_pregunta = 0
    respuestas = []
    conteo = {k: 0 for k in conteo}
    boton_siguiente.disabled = False
    salida.clear_output()
    mostrar_pregunta()

boton_reiniciar = widgets.Button(description="🔄 Reiniciar")
boton_reiniciar.on_click(reiniciar)

# Conectar eventos
boton_siguiente.on_click(siguiente)

# Mostrar
mostrar_pregunta()
display(barra, label_pregunta, radio_respuesta, boton_siguiente, salida, boton_reiniciar)